# Contextual Multi-Armed Bandit

For the contextual multi-armed bandit (cMAB) when user information is available (context), we implemented a generalisation of Thompson sampling algorithm ([Agrawal and Goyal, 2014](https://arxiv.org/pdf/1209.3352.pdf)) based on NumPyro.

![title](img/cmab.png)

The following notebook contains an example of usage of the class Cmab, which implements the algorithm above.

In [1]:
import numpy as np

from pybandits.cmab import CmabBernoulli
from pybandits.model import BayesianLogisticRegression, BnnLayerParams, BnnParams, FeaturesConfig, StudentTArray

/home/runner/.cache/pypoetry/virtualenvs/pybandits-vYJB-miV-py3.10/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
n_samples = 1000
n_features = 5

First, we need to define the input context matrix $X$ of size ($n\_samples, n\_features$) and the mapping of possible actions $a_i \in A$ to their associated model.

In [3]:
# context
X = 2 * np.random.random_sample((n_samples, n_features)) - 1  # random float in the interval (-1, 1)
print("X: context matrix of shape (n_samples, n_features)")
print(X[:10])

X: context matrix of shape (n_samples, n_features)
[[-0.51270902  0.99290505 -0.57389453  0.66379473 -0.66467283]
 [-0.5354891  -0.33973719 -0.76223006 -0.78500726 -0.22727163]
 [-0.49465275 -0.07163728 -0.16766756 -0.64453275 -0.35750767]
 [-0.99884928 -0.63582666 -0.30939484  0.78094937 -0.4590479 ]
 [ 0.17281302  0.19282414  0.81678002  0.07735765  0.90226522]
 [ 0.57300374  0.91440687  0.33897638 -0.49780777 -0.71092441]
 [-0.15592377 -0.62943931 -0.95421001  0.50065244  0.10287689]
 [-0.75458849  0.74273738  0.91914393 -0.90514371  0.44257554]
 [ 0.8962148   0.11878674 -0.06746334 -0.68992531 -0.0739651 ]
 [-0.84167319  0.52856973 -0.91637984  0.59285441  0.18347684]]


In [4]:
# define action model
bias = StudentTArray.cold_start(mu=1, sigma=2, shape=1)
weight = StudentTArray.cold_start(shape=(n_features, 1))
layer_params = BnnLayerParams(weight=weight, bias=bias)
model_params = BnnParams(bnn_layer_params=[layer_params])
feature_config = FeaturesConfig(n_features=n_features)

update_method = "VI"
update_kwargs = {"num_steps": 100, "batch_size": 128, "optimizer_type": "adam"}

actions = {
    "a1": BayesianLogisticRegression(
        model_params=model_params,
        feature_config=feature_config,
        update_method=update_method,
        update_kwargs=update_kwargs,
    ),
    "a2": BayesianLogisticRegression(
        model_params=model_params,
        feature_config=feature_config,
        update_method=update_method,
        update_kwargs=update_kwargs,
    ),
}

We can now init the bandit given the mapping of actions $a_i$ to their model.

In [5]:
# init contextual Multi-Armed Bandit model
cmab = CmabBernoulli(actions=actions)

The predict function below returns the action selected by the bandit at time $t$: $a_t = argmax_k P(r=1|\beta_k, x_t)$. The bandit selects one action per each sample of the contect matrix $X$.

In [6]:
# predict action
pred_actions, _, _ = cmab.predict(X)
print("Recommended action: {}".format(pred_actions[:10]))

Recommended action: ['a2', 'a2', 'a1', 'a2', 'a2', 'a2', 'a1', 'a1', 'a2', 'a1']

Now, we observe the rewards and the context from the environment. In this example rewards and the context are randomly simulated.

In [7]:
# simulate reward from environment
simulated_rewards = np.random.randint(2, size=n_samples).tolist()
print("Simulated rewards: {}".format(simulated_rewards[:10]))

Simulated rewards: [1, 0, 1, 0, 0, 1, 0, 1, 1, 1]


Finally, we update the model providing per each action sample: (i) its context $x_t$ (ii) the action $a_t$ selected by the bandit, (iii) the corresponding reward $r_t$.

In [8]:
# update model
cmab.update(context=X, actions=pred_actions, rewards=simulated_rewards)

SVI:   0%|          | 0/34 [00:00<?, ?it/s]

SVI:   3%|▎         | 1/34 [00:01<00:35,  1.08s/it]

SVI:   3%|▎         | 1/34 [00:01<00:35,  1.08s/it, loss=950.6215]

SVI:   6%|▌         | 2/34 [00:01<00:34,  1.08s/it, loss=659.3480]

SVI:   9%|▉         | 3/34 [00:01<00:33,  1.08s/it, loss=792.0102]

...

SVI:  91%|█████████ | 31/34 [00:00<00:02,  1.21it/s, loss=590.7429]

SVI:  94%|█████████▍| 32/34 [00:00<00:01,  1.21it/s, loss=483.9359]

SVI:  97%|█████████▋| 33/34 [00:00<00:00,  1.21it/s, loss=504.3119]

SVI: 100%|██████████| 34/34 [00:01<00:00, 24.51it/s, loss=504.3119]

SVI: 100%|██████████| 34/34 [00:01<00:00, 24.51it/s, loss=529.4441]